In [0]:
%run ./Classroom-Setup-Common

In [0]:
-- 4.2 Demo writes to the instructor_interop_demo catalog (customer-managed
-- external location). That catalog cannot be created from here because it
-- requires a MANAGED LOCATION pointing at an instructor-controlled S3 path.
-- Provision it once via the 3.2 demo setup (Includes/Classroom-Setup-3-demo)
-- before running 4.2.
BEGIN
  DECLARE catalog_exists BOOLEAN DEFAULT FALSE;

  SET catalog_exists = (
    SELECT COUNT(*) > 0
    FROM system.information_schema.catalogs
    WHERE catalog_name = 'instructor_interop_demo'
  );

  IF NOT catalog_exists THEN
    SELECT raise_error(
      'Catalog "instructor_interop_demo" not found. ' ||
      'Run the 3.2 demo setup notebook (Includes/Classroom-Setup-3-demo) before running this notebook.'
    );
  END IF;
END;

In [0]:
USE CATALOG instructor_interop_demo;

In [0]:
-- Schema is idempotent - safe to re-run.
CREATE SCHEMA IF NOT EXISTS data_interoperability_tpcds;
USE SCHEMA data_interoperability_tpcds;

In [0]:
-- 4.2 mutates this table (INSERT / UPDATE / MERGE), so drop and recreate every
-- run for deterministic verification. Source slice is small: two consecutive
-- TPC-DS sales dates (ss_sold_date_sk 2450816-2450817) for items 1-2000. This
-- guarantees the keys 4.2 will mutate (date 2450816, items 1000-1010) are
-- present, while keeping the table small enough that the EMR mutation step
-- stays snappy.
DROP TABLE IF EXISTS store_sales_iceberg;

CREATE TABLE store_sales_iceberg
USING ICEBERG
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'false',
  'delta.enableRowTracking'     = 'false'
)
AS
SELECT *
FROM samples.tpcds_sf1000.store_sales
WHERE ss_sold_date_sk BETWEEN 2450816 AND 2450817
  AND ss_item_sk      BETWEEN 1       AND 2000;

In [0]:
-- Surface the starting state so 4.2's verification step has a known baseline.
SELECT
  current_catalog() AS catalog,
  current_schema()  AS schema,
  (SELECT COUNT(*) FROM store_sales_iceberg) AS total_rows,
  (SELECT COUNT(*) FROM store_sales_iceberg
    WHERE ss_sold_date_sk = 2450816
      AND ss_item_sk BETWEEN 1000 AND 1010) AS mutation_slice_rows;